<a href="https://colab.research.google.com/github/Fazal2204/IMLcodingquestions/blob/main/Question1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Question 1a.

In [1]:
import numpy as np

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature=feature
        self.threshold=threshold
        self.left=left
        self.right=right
        self.value=value

class DecisionTree:
    def __init__(self,max_depth=10,min_samples_split=2,n_features=None):
        self.max_depth=max_depth
        self.min_samples_split=min_samples_split
        self.n_features=n_features
        self.root=None

    def build_tree(self,X,y,depth=0):
        if len(X)==0:
            return Node(value=0)
        n_samples=len(X)
        n_feats=len(X[0])
        if depth>=self.max_depth or n_samples<self.min_samples_split or len(set(y))==1:
            return Node(value=np.mean(y))
        feat_idxs=np.random.choice(n_feats,self.n_features,replace=False)
        best_feat,best_thresh=self._best_split(X,y,feat_idxs)
        if best_feat is None:
            return Node(value=np.mean(y))
        left_idxs,right_idxs=self._split(X,best_feat,best_thresh)
        if len(left_idxs)==0 or len(right_idxs)==0:
            return Node(value=np.mean(y))
        left=self.build_tree([X[i] for i in left_idxs],[y[i] for i in left_idxs],depth+1)
        right=self.build_tree([X[i] for i in right_idxs],[y[i] for i in right_idxs],depth+1)
        return Node(best_feat,best_thresh,left,right)

    def _best_split(self,X,y,feat_idxs):
        best_gain=-1
        split_idx=None
        split_thresh=None
        for feat_idx in feat_idxs:
            X_column=[row[feat_idx] for row in X]
            thresholds=set(X_column)
            for thr in thresholds:
                gain=self._variance_reduction(X,y,feat_idx,thr)
                if gain>best_gain:
                    best_gain=gain
                    split_idx=feat_idx
                    split_thresh=thr
        return split_idx,split_thresh

    def _variance_reduction(self,X,y,split_idx,threshold):
        parent_var=np.var(y)
        left_idxs,right_idxs=self._split(X,split_idx,threshold)
        if len(left_idxs)==0 or len(right_idxs)==0:
            return 0
        n=len(y)
        n_l=len(left_idxs)
        n_r=len(right_idxs)
        var_l=np.var([y[i] for i in left_idxs])
        var_r=np.var([y[i] for i in right_idxs])
        child_var=(n_l/n)*var_l+(n_r/n)*var_r
        return parent_var-child_var

    def _split(self,X,split_idx,threshold):
        left_idxs=[i for i,row in enumerate(X) if row[split_idx]<=threshold]
        right_idxs=[i for i,row in enumerate(X) if row[split_idx]>threshold]
        return left_idxs,right_idxs

    def fit(self,X,y):
        if self.n_features is None:
            self.n_features=X.shape[1]
        X_list=X.tolist()
        y_list=y.tolist()
        self.root=self.build_tree(X_list,y_list)

    def predict(self,x):
        node=self.root
        while node.value is None:
            if x[node.feature]<=node.threshold:
                node=node.left
            else:
                node=node.right
        return node.value

    def predict_batch(self,X):
        return np.array([self.predict(x) for x in X])

This code sets up the decision tree to recursively find splits that lower the MSE. It also includes checks for pure nodes or max depth to stop the recursion properly and prevent overfitting.

Question 1b.

In [2]:
class RandomForest:
    def __init__(self,n_trees=20,max_depth=10,min_samples_split=2,n_features=None):
        self.n_trees=n_trees
        self.max_depth=max_depth
        self.min_samples_split=min_samples_split
        self.n_features=n_features
        self.trees=[]

    def build_forest(self,X,y):
        self.trees=[]
        for _ in range(self.n_trees):
            tree=DecisionTree(max_depth=self.max_depth,
                              min_samples_split=self.min_samples_split,
                              n_features=self.n_features)
            X_samp,y_samp=self._bootstrap_samples(X,y)
            tree.fit(X_samp,y_samp)
            self.trees.append(tree)

    def _bootstrap_samples(self,X,y):
        n_samples=X.shape[0]
        idxs=np.random.choice(n_samples,n_samples,replace=True)
        return X[idxs],y[idxs]

    def predict(self,X):
        predictions=np.array([tree.predict_batch(X) for tree in self.trees])
        predictions=np.swapaxes(predictions,0,1)
        return np.array([np.mean(pred) for pred in predictions])

The random forest groups multiple decision trees together, using data bootstrapping and random feature selection to decorrelate them. Taking the mean of all these trees makes the model much more robust against noise

Question 1c.

In [3]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X,y=load_diabetes(return_X_y=True)

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

single_tree=DecisionTree(max_depth=10)
single_tree.fit(X_train,y_train)
tree_preds=single_tree.predict_batch(X_test)
tree_mse=mean_squared_error(y_test,tree_preds)
print("Single Decision Tree MSE:",tree_mse)

rf=RandomForest(n_trees=20,max_depth=10,n_features=int(np.sqrt(X.shape[1])))
rf.build_forest(X_train,y_train)
rf_preds=rf.predict(X_test)
rf_mse=mean_squared_error(y_test,rf_preds)
print("Random Forest MSE:",rf_mse)

Single Decision Tree MSE: 4294.3881228526325
Random Forest MSE: 3267.7408263854177


Testing on the diabetes dataset shows that the single Decision Tree tends to overfit and memorize the training data. In contrast, the Random Forest drastically lowers this variance through bagging, resulting in a much better test score.
